# Bonsai Image — Colab + serve.sh + localtunnel\nRun once. Prints the local and tunnel URLs.

In [ ]:
# setup + serve + tunnel\nimport os, subprocess, json, time, threading\nfrom pathlib import Path\n\nROOT = Path('.').resolve()\nREPO = ROOT / 'Bonsai-Image-Demo'\nURL_FILE = Path('/tmp/bonsai_url.json')\n\ndef run(cmd, capture=False, cwd=None):\n    p = subprocess.run(cmd, shell=True, text=True, capture_output=capture, cwd=cwd)\n    if capture:\n        return p.returncode, p.stdout, p.stderr\n    if p.stdout:\n        print(p.stdout.rstrip())\n    if p.returncode != 0:\n        print(f'FAILED ({p.returncode}): {cmd}')\n        if p.stderr:\n            print(p.stderr.rstrip())\n    return p.returncode\n\n# 1) clone\nif not REPO.exists():\n    run('git clone https://github.com/PrismML-Eng/Bonsai-Image-Demo.git')\n\n# 2) setup\nrun(f'chmod +x {REPO}/setup.sh')\nrun(f'cd {REPO} && ./setup.sh')\n\n# 3) start serve.sh in background\nprint('\nStarting serve.sh...')\nserve_proc = subprocess.Popen(\n    ['bash', f'{REPO}/scripts/serve.sh'],\n    cwd=str(REPO),\n    stdout=subprocess.PIPE,\n    stderr=subprocess.STDOUT,\n)\n\ndef _pipe(pipe):\n    for line in iter(pipe.readline, b''):\n        try:\n            print(line.decode('utf-8', errors='replace').rstrip())\n        except Exception:\n            pass\n    pipe.close()\n\nthreading.Thread(target=_pipe, args=(serve_proc.stdout,), daemon=True).start()\n\n# 4) wait for backend\nimport requests as _req\nready = False\nfor i in range(120):\n    try:\n        r = _req.get('http://127.0.0.1:8000/backends', timeout=2)\n        if r.status_code == 200:\n            print(f'Backend ready after {i}s')\n            ready = True\n            break\n    except Exception:\n        pass\n    if i % 10 == 0:\n        print(f'Waiting... ({i}s)')\n    time.sleep(1)\n\nif not ready:\n    print('WARNING: backend did not become ready on port 8000')\n\n# 5) expose with localtunnel (installed by repo setup via Node.js)\nprint('\nStarting localtunnel on port 8000...')\nlt = subprocess.Popen(\n    [str(REPO / '.venv' / 'bin' / 'lt'), '--port', '8000'],\n    cwd=str(REPO),\n    stdout=subprocess.PIPE,\n    stderr=subprocess.STDOUT,\n    text=True,\n)\n\nlt_url = None\nfor _ in range(90):\n    line = lt.stdout.readline() if lt.stdout else ''\n    if not line:\n        time.sleep(1)\n        continue\n    print('localtunnel:', line.rstrip())\n    if 'your url is:' in line.lower() or 'https://' in line.lower():\n        parts = line.strip().split()\n        for p in parts:\n            if p.startswith('https://'):\n                lt_url = p\n                break\n    if lt_url:\n        break\n\nprint('\n' + '='*60)\nprint('READY')\nprint(f'Local URL:      http://127.0.0.1:8000')\nprint(f'Tunnel URL:     {lt_url}')\nprint(f'Generate:       {lt_url}/generate if lt_url else \"N/A\"}')\nprint('='*60)\n\nURL_FILE.write_text(json.dumps({'local_url': 'http://127.0.0.1:8000', 'tunnel_url': lt_url}))\n